# 核心概念

## 组件指南
本指南概述了 Ragas 内部使用的不同组件。

- 提示对象
- 评估样本
- 评估数据集

### 提示对象
Ragas 中的提示符可用于各种指标和合成数据生成任务。在每个任务中，Ragas 还允许用户修改默认提示符或将其替换为自定义提示符。本指南概述了 Ragas 中的提示符对象。

提示对象的组成部分
在 Ragas 中，提示对象由以下关键组件组成：

1. 指令：指令是任何提示的基本元素，它是一种自然语言指令，清晰地描述了语言模型 (LLM) 应执行的任务。指令通过instruction提示对象中的变量来指定。

2. 少量样本示例：众所周知，LLM 在提供少量样本示例时表现更佳，因为它们有助于模型理解任务上下文并生成更准确的响应。这些示例使用examples提示对象中的变量指定。每个示例都包含一个输入及其对应的输出，LLM 会使用它们来学习任务。

3. 输入模型：每个提示都需要输入来产生输出。在 Ragas 中，此输入的预期格式使用input_model变量定义。这是一个 Pydantic 模型，它概述了输入的结构，从而可以验证和解析提供给提示的数据。

4. 输出模型：执行后，提示会生成输出。此输出的格式由output_model提示对象中的变量指定。与输入模型类似，输出模型是一个 Pydantic 模型，它定义了输出的结构，方便验证和解析 LLM 生成的数据。

#### 例子
下面是一个定义文本生成任务提示的提示对象的示例：


In [ ]:
from ragas.prompt.pydantic_prompt import PydanticPrompt
from pydantic import BaseModel, Field

class MyInput(BaseModel):
    question: str = Field(description="The question to answer")

class MyOutput(BaseModel):
    answer: str = Field(description="The answer to the question")

class MyPrompt(PydanticPrompt[MyInput,MyOutput]):
    instruction = "Answer the given question"
    input_model = MyInput
    output_model = MyOutput
    examples = [
        (
            MyInput(question="Who's building the opensource standard for LLM app evals?"),
            MyOutput(answer="Ragas")
        )
    ]

### 创建有效提示的指南
在 Ragas 中创建提示时，请考虑以下准则，以确保提示有效且符合任务要求：

1. 清晰简洁的指示：提供清晰简洁的指示，明确定义法学硕士 (LLM) 应执行的任务。指示含糊不清可能会导致答案不准确。
2. 相关的少量示例：包含与任务相关的各种场景（理想情况下为 3-5 个）的相关少量示例。这些示例有助于法学硕士 (LLM) 理解上下文并生成准确的答案。
3. 简单的输入和输出模型：定义简单直观的输入和输出模型，准确表示 LLM 所需的数据格式以及 LLM 生成的输出。如果模型比较复杂，请尝试将任务分解为更小的子任务，并分别设置提示。

### 评估样本
评估样本是一个单一的结构化数据实例，用于评估和衡量 LLM 应用程序在特定场景下的性能。它代表了 AI 应用程序预期处理的单个交互单元或特定用例。在 Ragas 中，评估样本使用`SingleTurnSample`和`MultiTurnSample`类表示。


#### 单样本
`SingleTurnSample` 表示用户、LLM 和预期评估结果之间的单轮交互。它适用于涉及单个问答对的评估，可能还会附加上下文或参考信息


以下示例演示了如何`SingleTurnSample`在基于 RAG 的应用程序中创建一个用于评估单轮互动的实例。在此场景中，用户提出一个问题，然后 AI 给出答案。我们将创建一个 `SingleTurnSample` 实例来表示此互动，其中包括检索到的上下文、参考答案和评估标准。



In [5]:
from ragas import SingleTurnSample

# User's question
user_input = "What is the capital of France?"

# Retrieved contexts (e.g., from a knowledge base or search engine)
retrieved_contexts = ["Paris is the capital and most populous city of France."]

# AI's response
response = "The capital of France is Paris."

# Reference answer (ground truth)
reference = "Paris"

# Evaluation rubric
rubric = {
    "accuracy": "Correct",
    "completeness": "High",
    "fluency": "Excellent"
}

# Create the SingleTurnSample instance
sample = SingleTurnSample(
    user_input=user_input,
    retrieved_contexts=retrieved_contexts,
    response=response,
    reference=reference,
    rubrics=rubric
)

In [6]:
sample.to_dict()

{'user_input': 'What is the capital of France?',
 'retrieved_contexts': ['Paris is the capital and most populous city of France.'],
 'response': 'The capital of France is Paris.',
 'reference': 'Paris',
 'rubrics': {'accuracy': 'Correct',
  'completeness': 'High',
  'fluency': 'Excellent'}}

#### 多样本
`MultiTurnSample` 表示人类、AI 以及可选工具之间的多轮交互，以及预期的评估结果。它适用于表示对话代理在更复杂的交互中进行评估。在 中`MultiTurnSampl`e，user_input属性 表示一系列消息，这些消息共同构成人类用户与 AI 系统之间的多轮对话。这些消息是类 `HumanMessage`、`AIMessage` 和 `ToolMessage` 的实例

##### 例子
以下示例演示了如何创建一个MultiTurnSample用于评估多轮交互的实例。在此场景中，用户想要了解纽约市的当前天气。AI 助手将使用天气 API 工具获取信息并响应用户。

In [ ]:
from ragas.messages import HumanMessage, AIMessage, ToolMessage, ToolCall

# User asks about the weather in New York City
user_message = HumanMessage(content="What's the weather like in New York City today?")

# AI decides to use a weather API tool to fetch the information
ai_initial_response = AIMessage(
    content="Let me check the current weather in New York City for you.",
    tool_calls=[ToolCall(name="WeatherAPI", args={"location": "New York City"})]
)

# Tool provides the weather information
tool_response = ToolMessage(content="It's sunny with a temperature of 75°F in New York City.")

# AI delivers the final response to the user
ai_final_response = AIMessage(content="It's sunny and 75 degrees Fahrenheit in New York City today.")

# Combine all messages into a list to represent the conversation
conversation = [
    user_message,
    ai_initial_response,
    tool_response,
    ai_final_response
]

现在，使用对话创建一个 MultiTurnSample 对象，包括任何参考响应和评估标准。

In [ ]:
from ragas import MultiTurnSample
# Reference response for evaluation purposes
reference_response = "Provide the current weather in New York City to the user."


# Create the MultiTurnSample instance
sample = MultiTurnSample(
    user_input=conversation,
    reference=reference_response,
)

### 评估数据集
评估数据集是一组同质的数据样本，旨在评估 AI 应用程序的性能和功能。在 Ragas 中，评估数据集使用EvaluationDataset类来表示，该类提供了一种结构化的方式来组织和管理用于评估目的的数据样本。


#### 概述
评估数据集的结构
评估数据集包括：

- Samples ： SingleTurnSample或MultiTurnSample实例的集合。每个样本代表一个独特的交互或场景。
- 一致性：数据集内的所有样本应属于同一类型（全部为单转样本或全部为多转样本），以保持评估的一致性。


#####  从 SingleTurnSamples 创建评估数据集
在此示例中，我们将演示如何使用多个SingleTurnSample实例创建评估数据集 (EvaluationDataset)。我们将逐步讲解整个过程，包括创建单个样本、将它们组合成数据集，以及对数据集执行基本操作。

- 步骤 1：导入必要的类

首先，从模块中导入 `SingleTurnSample` 和 `EvaluationDataset` 类。



In [ ]:
from ragas import SingleTurnSample, EvaluationDataset



- 步骤 2：创建单独的样本

创建多个代表单个评估样本的 SingleTurnSample 实例。

In [ ]:
# Sample 1
sample1 = SingleTurnSample(
    user_input="What is the capital of Germany?",
    retrieved_contexts=["Berlin is the capital and largest city of Germany."],
    response="The capital of Germany is Berlin.",
    reference="Berlin",
)

# Sample 2
sample2 = SingleTurnSample(
    user_input="Who wrote 'Pride and Prejudice'?",
    retrieved_contexts=["'Pride and Prejudice' is a novel by Jane Austen."],
    response="'Pride and Prejudice' was written by Jane Austen.",
    reference="Jane Austen",
)

# Sample 3
sample3 = SingleTurnSample(
    user_input="What's the chemical formula for water?",
    retrieved_contexts=["Water has the chemical formula H2O."],
    response="The chemical formula for water is H2O.",
    reference="H2O",
)

- 步骤 3：创建 EvaluationDataset 通过传递 SingleTurnSample 实例列表来创建 EvaluationDataset。


In [ ]:
dataset = EvaluationDataset(samples=[sample1, sample2, sample3])

##### 从 Hugging Face 数据集加载评估数据集

在实际操作中，您可能希望从现有数据集源（例如 Hugging Face Datasets 库）加载评估数据集。以下示例演示了如何从 Hugging Face 数据集加载评估数据集并将其转换为 EvaluationDataset 实例。

确保数据集包含评估所需的字段，例如用户输入、检索到的上下文、响应和参考。

In [ ]:
from datasets import load_dataset
dataset = load_dataset("explodinggradients/amnesty_qa","english_v3")

将数据集加载到 Ragas EvaluationDataset 对象中。

In [ ]:
from ragas import EvaluationDataset

eval_dataset = EvaluationDataset.from_hf_dataset(dataset["eval"])

## 指标
### 指标概述
指标是用于评估 AI 应用程序性能的定量指标。指标有助于评估应用程序及其各个组件相对于给定测试数据的性能。它们为整个应用程序开发和部署过程中的比较、优化和决策提供了数值基础。指标对于以下方面至关重要：

- 组件选择：指标可用于将 AI 应用程序的不同组件（如 LLM、检索器、代理配置等）与您自己的数据进行比较，并从不同的选项中选择最佳组件。
- 错误诊断和调试：指标有助于识别应用程序的哪个部分导致错误或性能不佳，从而更容易调试和改进。
- 持续监控和维护：指标可以跟踪 AI 应用程序随时间的性能，帮助检测和应对数据漂移、模型退化或用户需求变化等问题。


<img src="https://docs.ragas.io/en/stable/_static/imgs/metrics_mindmap.png">

根据底层使用的机制，指标可分为两类：
- 基于 LLM 的指标：这些指标使用底层的 LLM 进行评估。可能需要执行一次或多次 LLM 调用才能得出分数或结果。这些指标可能具有一定的不确定性，因为 LLM 可能不会总是对相同的输入返回相同的结果。另一方面，这些指标已被证明更准确，更接近人工评估。

ragas 中所有基于 LLM 的指标均继承自MetricWithLLM该类。这些指标需要在评分前设置一个 LLM 对象。

In [ ]:
from ragas.metrics import FactualCorrectness
scorer = FactualCorrectness(llm=evaluation_llm)

每个基于 LLM 的指标也将具有使用Prompt Object编写的与其相关的提示。

     非基于 LLM 的指标：这些指标不使用底层的 LLM 进行评估。这些指标是确定性的，无需使用 LLM 即可评估 AI 应用程序的性能。这些指标依赖于传统方法来评估 AI 应用程序的性能，例如字符串相似度、BLEU 分数等。因此，这些指标与人工评估的相关性较低。

ragas 中所有基于 LLM 的指标都是从Metric类继承而来的。

根据评估的数据类型，指标大致可分为两类：

     单回合指标：这些指标基于用户与 AI 之间的单回合交互来评估 AI 应用程序的性能。ragas 中所有支持单回合评估的指标都继承自SingleTurnMetric类，并使用single_turn_ascore方法进行评分。它还接受一个Single Turn Sample对象作为输入。

In [ ]:
from ragas.metrics import FactualCorrectness

scorer = FactualCorrectness()
await scorer.single_turn_ascore(sample)

     多轮指标：这些指标基于用户与 AI 之间的多轮交互来评估 AI 应用程序的性能。ragas 中所有支持多轮评估的指标都继承自MultiTurnMetric类，并使用multi_turn_ascore方法进行评分。它还接受一个Multi Turn Sample对象作为输入。

In [ ]:
from ragas.metrics import AgentGoalAccuracyWithReference
from ragas import MultiTurnSample

scorer = AgentGoalAccuracyWithReference()
await scorer.multi_turn_ascore(sample)

### 度量设计原则
为 AI 应用设计有效的指标需要遵循一系列核心原则，以确保其可靠性、可解释性和相关性。以下是我们在设计指标时遵循的五个关键原则：

1. 单一指标
应仅针对AI应用程序性能的某个特定方面。这确保了该指标既可解释又可操作，并能清晰地洞察被测对象。

2. 直观易懂的
指标设计应易于理解和解读。清晰直观的指标能够更轻松地传达结果并得出有意义的结论。

3. 有效的提示流程
使用大型语言模型 (LLM) 开发指标时，应使用与人工评估紧密结合的智能提示流程。将复杂任务分解为具有特定提示的较小子任务，可以提高指标的准确性和相关性。

4. 稳健性：
确保基于法学硕士 (LLM) 的指标包含足够多的少量样本，以反映预期结果。这为法学硕士 (LLM) 提供了可遵循的背景和指导，从而增强了指标的稳健性。

5. 一致的评分范围
规范化指标分数值或确保它们落在特定范围内（例如 0 到 1）至关重要。这有助于比较不同的指标，并有助于保持整个评估框架的一致性和可解释性。

这些原则为创建不仅有效而且实用且有意义的评估人工智能应用的指标奠定了基础。

### 可用指标列表
Ragas 提供了一组评估指标，可用于衡量 LLM 应用程序的性能。这些指标旨在帮助您客观地衡量应用程序的性能。这些指标适用于不同的应用程序和任务，例如 RAG 和 Agentic 工作流。

每个指标本质上都是用于评估应用程序特定方面的范例。基于 LLM 的指标可能使用一个或多个 LLM 调用来得出分数或结果。您也可以使用 ragas 修改或编写自己的指标。

#### 检索增强生成
- 上下文精度
- 上下文回忆
- 上下文实体回忆
- 噪声敏感度
- 响应相关性
- 忠诚
- 多模态忠实度
- 多模态相关性
#### Nvidia 指标
- 答案准确率
- 语境相关性
- 回应接地
#### 代理或工具用例
- 主题坚持
- 工具调用准确度
- 代理目标准确度
#### 自然语言比较
- 事实正确性
- 语义相似性
- 非 LLM 字符串相似性
- BLEU 分数
- ROUGE 分数
- 字符串存在
- 精确匹配
#### SQL
- 基于执行的数据合规分数
- SQL 查询等价性
#### 通用
- 方面评论家
- 简单标准评分
- 基于评分标准的评分
- 特定实例的评分标准
#### 其他任务
- 总结

#### 上下文精度

上下文准确率 (`Precision@k`) 是衡量上下文中相关词块比例的指标`retrieved_contexts`。它被计算为上下文中每个词块的准确率 (`Precision@k`) 的平均值。准确率是排名为 `k` 的相关词块数量与排名为 k 的词块总数之比。

$\begin{gathered}
\text { Context Precision@K }=\frac{\sum_{k=1}^K\left(\text { Precision@ } \mathrm{k} \times v_k\right)}{\text { Total number of relevant items in the top } K \text { results }} \\
\text { Precision@ } \mathrm{k}=\frac{\text { true positives@ } \mathrm{k}}{(\text { true positives@ } \mathrm{k}+\text { false positives } @ \mathrm{k})}
\end{gathered}$

##### 基于 LLM 的上下文精度**


以下指标使用 LLM 来识别检索到的上下文是否相关。

无参考上下文精度

`LLMContextPrecisionWithoutReference`当你同时拥有检索到的上下文和与 关联的引用上下文时，可以使用 指标`user_input`。为了评估 检索到的上下文是否相关，此方法使用 LLM 将 中存在的每个检索到的上下文或块`retrieved_contexts`与进行比较response。

In [7]:
from ragas import SingleTurnSample
from ragas.metrics import LLMContextPrecisionWithoutReference

context_precision = LLMContextPrecisionWithoutReference(llm=evaluator_llm)

sample = SingleTurnSample(
    user_input="Where is the Eiffel Tower located?",
    response="The Eiffel Tower is located in Paris.",
    retrieved_contexts=["The Eiffel Tower is located in Paris."], 
)


await context_precision.single_turn_ascore(sample)

NameError: name 'evaluator_llm' is not defined

输出

0.9999999999

##### 上下文精度与参考值
`LLMContextPrecisionWithReference`当你同时拥有检索到的上下文和与 关联的参考答案时，可以使用 指标`user_input`。为了评估 检索到的上下文是否相关，此方法使用 LLM 将 中存在的每个检索到的上下文或块retrieved_contexts与进行比较reference。

例子



In [ ]:
from ragas import SingleTurnSample
from ragas.metrics import LLMContextPrecisionWithReference

context_precision = LLMContextPrecisionWithReference(llm=evaluator_llm)

sample = SingleTurnSample(
    user_input="Where is the Eiffel Tower located?",
    reference="The Eiffel Tower is located in Paris.",
    retrieved_contexts=["The Eiffel Tower is located in Paris."], 
)

await context_precision.single_turn_ascore(sample)

输出

0.9999999999

#### 非基于 LLM 的上下文精度
该指标采用传统方法来确定检索到的上下文是否相关。它依赖非基于 LLM 的指标作为距离度量来评估检索到的上下文的相关性。

##### 参考上下文的上下文精度
该NonLLMContextPrecisionWithReference指标专为检索上下文和参考上下文都可用于 的场景而设计user_input。为了确定检索到的上下文是否相关，此方法使用非基于 LLM 的相似性度量，将retrieved_contexts 中每个检索到的上下文或块与 中的每个上下文进行比较。reference_contexts

In [ ]:
from ragas import SingleTurnSample
from ragas.metrics import NonLLMContextPrecisionWithReference

context_precision = NonLLMContextPrecisionWithReference()

sample = SingleTurnSample(
    retrieved_contexts=["The Eiffel Tower is located in Paris."], 
    reference_contexts=["Paris is the capital of France.", "The Eiffel Tower is one of the most famous landmarks in Paris."]
)

await context_precision.single_turn_ascore(sample)

输出

0.9999999999

#### 上下文回忆
上下文召回率衡量成功检索到的相关文档（或信息片段）数量。它关注的是不要遗漏重要的结果。更高的召回率意味着遗漏的相关文档更少。简而言之，召回率就是不要遗漏任何重要的内容。正因为要确保不遗漏任何内容，计算上下文召回率总是需要一个参考来进行比较。

基于法学硕士的上下文回忆
LLMContextRecall使用和 计算 user_input，值在 0 到 1 之间，值越高表示性能越好。此指标用作代理，也使其更容易使用，因为注释参考上下文可能非常耗时。为了估计上下文召回率，将参考分解为声明，并分析答案中的每个声明以确定它是否可以归因于检索到的上下文。在理想情况下，参考答案中的所有声明都应归因于检索到的上下文。referenceretrieved_contextsreferencereference_contextsreferencereference

上下文召回率的计算公式如下：

$\text { Context Recall }=\frac{\text { Number of claims in the reference supported by the retrieved context }}{\text { Total number of claims in the reference }}$

In [ ]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import LLMContextRecall

sample = SingleTurnSample(
    user_input="Where is the Eiffel Tower located?",
    response="The Eiffel Tower is located in Paris.",
    reference="The Eiffel Tower is located in Paris.",
    retrieved_contexts=["Paris is the capital of France."], 
)

context_recall = LLMContextRecall(llm=evaluator_llm)
await context_recall.single_turn_ascore(sample)

输出

1.0


##### 非法学硕士的上下文回忆
NonLLMContextRecall指标使用retrieved_contexts和reference_contexts计算，值在 0 到 1 之间，值越高表示性能越好。此指标使用非 LLM 字符串比较指标来识别检索到的上下文是否相关。您可以使用任何非 LLM 指标作为距离度量来识别检索到的上下文是否相关。

上下文召回率的计算公式如下：

$\text { context recall }=\frac{\mid \text { Number of relevant contexts retrieved } \mid}{\mid \text { Total number of reference contexts } \mid}$

In [ ]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import NonLLMContextRecall

sample = SingleTurnSample(
    retrieved_contexts=["Paris is the capital of France."], 
    reference_contexts=["Paris is the capital of France.", "The Eiffel Tower is one of the most famous landmarks in Paris."]
)

context_recall = NonLLMContextRecall()
await context_recall.single_turn_ascore(sample)

输出

0.5

#### 上下文实体召回
上下文实体召回
`ContextEntityRecall`指标给出了检索到的上下文的召回率，该指标基于 和 中同时存在的实体数量reference相retrieved_contexts对于 中单独存在的实体数量的比率reference。简而言之，它衡量了从 中召回的实体比例reference。该指标在基于事实的用例中很有用，例如旅游帮助台、历史问答等。该指标可以帮助评估实体的检索机制，它基于与 中存在的实体的比较，reference因为在实体很重要的情况下，我们需要能够`retrieved_contexts`覆盖它们的 。

为了计算这个指标，我们使用两组数据：
- RE:：引用中的实体集。
- RCE：检索到的上下文中的实体集。

我们计算两个集合中共有的实体数量（$R C E \cap R E$) 并将其除以参考中的实体总数 (RE）。公式为：

$\text { Context Entity Recall }=\frac{\text { Number of common entities between } R C E \text { and } R E}{\text { Total number of entities in } R E}$

In [ ]:
from ragas import SingleTurnSample
from ragas.metrics import ContextEntityRecall

sample = SingleTurnSample(
    reference="The Eiffel Tower is located in Paris.",
    retrieved_contexts=["The Eiffel Tower is located in Paris."], 
)

scorer = ContextEntityRecall(llm=evaluator_llm)

await scorer.single_turn_ascore(sample)

输出

0.999999995

如何计算

例子

> 参考：泰姬陵是一座象牙白色的大理石陵墓，位于印度阿格拉市亚穆纳河右岸。它是由莫卧儿皇帝沙贾汗于 1631 年下令建造的，用于安葬他最喜爱的妻子穆塔兹·玛哈尔。 高实体召回上下文：泰姬陵是爱情的象征，也是位于印度阿格拉的建筑奇迹。它由莫卧儿皇帝沙贾汗为纪念他心爱的妻子穆塔兹·玛哈尔而建造。该建筑以其复杂的大理石工艺和周围美丽的花园而闻名。 低实体召回上下文：泰姬陵是印度的标志性纪念碑。它是联合国教科文组织世界遗产，每年吸引数百万游客。复杂的雕刻和令人惊叹的建筑使它成为一个必游之地。

让我们考虑一下上面给出的参考和检索到的上下文。

- 步骤 1：查找参考中存在的实体。
    - 基本事实中的实体（RE） - ['Taj Mahal', 'Yamuna', 'Agra', '1631', 'Shah Jahan', 'Mumtaz Mahal']
- 步骤 2：查找检索到的上下文中存在的实体。
    - 上下文中的实体 (RCE1) - ['泰姬陵', '阿格拉', '沙贾汗', '穆塔兹·玛哈尔', '印度']
    - 上下文中的实体 (RCE2) - ['泰姬陵', '联合国教科文组织', '印度']
- 步骤 3：使用上面给出的公式计算实体召回率


$\begin{gathered}
\text { context entity recall } 1=\frac{|R C E 1 \cap R E|}{|R E|}=4 / 6=0.666 \\
\qquad \text { context entity recall } 2=\frac{|R C E 2 \cap R E|}{|R E|}=1 / 6
\end{gathered}$

我们可以看到，第一个上下文具有较高的实体召回率，因为它在参考文献中具有更好的实体覆盖率。如果用两种检索机制在同一组文档上获取这两个检索到的上下文，我们可以说，在实体重要的用例中，第一种机制优于另一种机制。

#### 噪声敏感度

NoiseSensitivity衡量系统在使用相关或不相关的检索文档时，给出错误响应的频率。分数范围从 0 到 1，值越低表示性能越好。噪声敏感度使用user_input、 reference、response和计算retrieved_contexts。

为了评估噪声敏感度，需要检查生成的回复中的每个断言，以确定其是否基于基本事实 (ground truth) 正确，以及是否可以归因于相关（或不相关）的检索上下文。理想情况下，答案中的所有断言都应得到相关检索上下文的支持。

$\text { noise sensitivity (relevant) }=\frac{\mid \text { Total number of incorrect claims in response } \mid}{\mid \text { Total number of claims in the response } \mid}$

In [ ]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import NoiseSensitivity

sample = SingleTurnSample(
    user_input="What is the Life Insurance Corporation of India (LIC) known for?",
    response="The Life Insurance Corporation of India (LIC) is the largest insurance company in India, known for its vast portfolio of investments. LIC contributes to the financial stability of the country.",
    reference="The Life Insurance Corporation of India (LIC) is the largest insurance company in India, established in 1956 through the nationalization of the insurance industry. It is known for managing a large portfolio of investments.",
    retrieved_contexts=[
        "The Life Insurance Corporation of India (LIC) was established in 1956 following the nationalization of the insurance industry in India.",
        "LIC is the largest insurance company in India, with a vast network of policyholders and huge investments.",
        "As the largest institutional investor in India, LIC manages substantial funds, contributing to the financial stability of the country.",
        "The Indian economy is one of the fastest-growing major economies in the world, thanks to sectors like finance, technology, manufacturing etc."
    ]
)

scorer = NoiseSensitivity(llm=evaluator_llm)
await scorer.single_turn_ascore(sample)

输出


0.3333333333333333

为了计算不相关上下文的噪声敏感度，可以将mode参数设置为irrelevant。





In [ ]:
scorer = NoiseSensitivity(mode="irrelevant")
await scorer.single_turn_ascore(sample)

例子

问题：印度人寿保险公司 (LIC) 以什么闻名？

事实：印度人寿保险公司（LIC）是印度最大的保险公司，成立于1956年，由保险业国有化而来。该公司以管理庞大的投资组合而闻名。

相关检索： - 印度人寿保险公司 (LIC) 成立于 1956 年，紧随印度保险业国有化之后。 - LIC 是印度最大的保险公司，拥有庞大的保单持有人网络，在金融领域发挥着重要作用。 - 作为印度最大的机构投资者，LIC 管理着大量的人寿基金，为该国的金融稳定做出了贡献。

不相关检索： - 得益于金融、技术、制造业等领域的发展，印度经济是世界上增长最快的主要经济体之一。


让我们来看看如何计算相关上下文中的噪声敏感度：

- 步骤 1：确定可以推断基本事实的相关背景。
- 事实：印度人寿保险公司（LIC）是印度最大的保险公司，成立于1956年，由保险业国有化而来。该公司以管理庞大的投资组合而闻名。
    - 上下文：
        - 背景 1：印度人寿保险公司 (LIC) 成立于 1956 年，紧随印度保险业国有化之后。
        - 背景 2：LIC 是印度最大的保险公司，拥有庞大的保单持有人网络，在金融领域发挥着重要作用。
        - 背景3：作为印度最大的机构投资者，LIC管理着大量资金，为该国的金融稳定做出了贡献。
- 第 2 步：验证生成的答案中的声明是否可以从相关上下文中推断出来。
    - 答：印度人寿保险公司（LIC）是印度最大的保险公司，以其庞大的投资组合而闻名。LIC为印度的金融稳定做出了贡献。
    - 上下文：
        - 背景 1：印度人寿保险公司 (LIC) 成立于 1956 年，紧随印度保险业国有化之后。
        - 背景 2：LIC 是印度最大的保险公司，拥有庞大的保单持有人网络，在金融领域发挥着重要作用。
        - 背景3：作为印度最大的机构投资者，LIC管理着大量资金，为该国的金融稳定做出了贡献。
- 步骤 3：识别答案中的任何不正确的声明（即，不受基本事实支持的答案陈述）。
    - 事实：印度人寿保险公司（LIC）是印度最大的保险公司，成立于1956年，由保险业国有化而来。该公司以管理庞大的投资组合而闻名。
    - 答：印度人寿保险公司（LIC）是印度最大的保险公司，以其庞大的投资组合而闻名。LIC为印度的金融稳定做出了贡献。

解释：基本事实并未提及 LIC 对国家金融稳定的贡献。因此，答案中的这一表述是错误的。

错误陈述：1 索赔总数：3

- 步骤 4：使用以下公式计算噪声敏感度：

 $\text { noise sensitivity }=\frac{1}{3}=0.333$

 这导致噪声敏感度得分为 0.333，表明答案中的三分之一声明是不正确的。


#### 响应相关性
响应相关性
该ResponseRelevancy指标衡量响应与用户输入的相关性。分数越高，表示与用户输入的匹配度越高；如果响应不完整或包含冗余信息，则分数越低。

该指标使用user_input和计算response如下：
- 根据回答生成一组人工问题（默认为 3 个）。这些问题旨在反映回答的内容。
- 计算用户输入的嵌入之间的余弦相似度（$\left(E_o\right)$) 以及每个生成问题的嵌入 ($\left(E_{g_{i}}\right)$）。
- 取这些余弦相似度得分的平均值来获得答案相关性：

$\begin{gathered}
\text { Answer Relevancy }=\frac{1}{N} \sum_{i=1}^N \operatorname{cosine} \operatorname{similarity}\left(E_{g_i}, E_o\right) \\
\text { Answer Relevancy }=\frac{1}{N} \sum_{i=1}^N \frac{E_{g_i} \cdot E_o}{\left\|E_{g_i}\right\|\left\|E_o\right\|}
\end{gathered}$

- $E_{g_i}$ ：嵌入 $i^{t h}$ 生成的问题
- $E_o$ ：嵌入用户输入。
- $N$ ：生成的问题数量（默认为3）。

注意：虽然分数通常在 0 到 1 之间，但由于余弦相似度的数学范围是 -1 到 1，因此无法保证。

如果答案直接且恰当地解答了原始问题，则该答案被视为相关。此指标侧重于答案与问题意图的契合程度，而非事实准确性。它会对不完整或包含不必要细节的答案进行惩罚。

In [ ]:
from ragas import SingleTurnSample 
from ragas.metrics import ResponseRelevancy

sample = SingleTurnSample(
        user_input="When was the first super bowl?",
        response="The first superbowl was held on Jan 15, 1967",
        retrieved_contexts=[
            "The First AFL–NFL World Championship Game was an American football game played on January 15, 1967, at the Los Angeles Memorial Coliseum in Los Angeles."
        ]
    )

scorer = ResponseRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings)
await scorer.single_turn_ascore(sample)

输出

0.9165088378587264

例子

问题：法国在哪里？它的首都是哪里？

相关性低的答案：法国位于西欧。

高度相关答案：法国位于西欧，巴黎是其首都。



为了计算答案与给定问题的相关性，我们遵循两个步骤：
- 步骤 1：使用大型语言模型 (LLM)，根据生成的答案对问题的 n 个变体进行逆向工程。例如，对于第一个答案，LLM 可能会生成以下可能的问题：
    - 问题 1： “法国位于欧洲哪个地区？”
    - 问题2： “法国在欧洲的地理位置是什么？”
    - 问题 3： “你能确定法国位于欧洲哪个地区吗？”
- 步骤2：计算生成问题与实际问题的平均余弦相似度。

其基本概念是，如果答案正确地回答了问题，那么很有可能仅凭答案就可以重建原始问题。


#### 忠诚

忠诚
忠诚度response指标衡量的是 a 与 的事实一致性retrieved context。其范围从 0 到 1，分数越高，一致性越好。

如果答复的所有主张都能得到检索到的上下文的支持， 则该答复被视为忠实的。

计算方法：
1. 识别回复中的所有声明。
2. 检查每个声明，看看是否可以从检索到的上下文中推断出来。
3. 使用以下公式计算忠实度得分：

$\text { Faithfulness Score }=\frac{\text { Number of claims in the response supported by the retrieved context }}{\text { Total number of claims in the response }}$


In [ ]:
from ragas.dataset_schema import SingleTurnSample 
from ragas.metrics import Faithfulness

sample = SingleTurnSample(
        user_input="When was the first super bowl?",
        response="The first superbowl was held on Jan 15, 1967",
        retrieved_contexts=[
            "The First AFL–NFL World Championship Game was an American football game played on January 15, 1967, at the Los Angeles Memorial Coliseum in Los Angeles."
        ]
    )
scorer = Faithfulness(llm=evaluator_llm)
await scorer.single_turn_ascore(sample)

##### 忠实于 HHEM-2.1-Open
Vectara 的 HHEM-2.1-Open是一个分类器模型 (T5)，经过训练可以从 LLM 生成的文本中检测幻觉。该模型可用于计算忠实度的第二步，即将声明与给定上下文进行交叉核对，以确定其是否可以从上下文推断出来。该模型免费、小型且开源，因此在生产用例中非常高效。要使用该模型计算忠实度，您可以使用以下代码片段：

In [ ]:
from ragas.dataset_schema import SingleTurnSample 
from ragas.metrics import FaithfulnesswithHHEM


sample = SingleTurnSample(
        user_input="When was the first super bowl?",
        response="The first superbowl was held on Jan 15, 1967",
        retrieved_contexts=[
            "The First AFL–NFL World Championship Game was an American football game played on January 15, 1967, at the Los Angeles Memorial Coliseum in Los Angeles."
        ]
    )
scorer = FaithfulnesswithHHEM(llm=evaluator_llm)
await scorer.single_turn_ascore(sample)

您可以通过设置参数将模型加载到指定的设备上device，并使用参数调整推理的批次大小batch_size。默认情况下，模型以 10 的批次大小加载到 CPU 上

In [ ]:
my_device = "cuda:0"
my_batch_size = 10

scorer = FaithfulnesswithHHEM(device=my_device, batch_size=my_batch_size)
await scorer.single_turn_ascore(sample)

例子

问题：爱因斯坦出生于何时何地？

背景：阿尔伯特·爱因斯坦（生于 1879 年 3 月 14 日）是一位德国出生的理论物理学家，被广泛认为是有史以来最伟大、最具影响力的科学家之一
    - 高忠诚度回答：爱因斯坦于1879年3月14日出生于德国。
    - 低忠诚度答案：爱因斯坦于 1879 年 3 月 20 日出生于德国。


让我们来看看如何使用低忠诚度答案来计算忠诚度：

- 步骤 1：将生成的答案分解为单独的语句。

- 声明：
    - 陈述 1：“爱因斯坦出生于德国。”
    - 陈述 2：“爱因斯坦出生于 1879 年 3 月 20 日。”
- 第 2 步：对于生成的每个语句，验证是否可以从给定的上下文推断出来。
    - 声明 1：是的
    - 声明2：否
- 步骤3：使用上面的公式计算忠诚度。

$\text { Faithfulness }=\frac{1}{2}=0.5$